# פרק 7: בניית יישומי צ'אט
## התחלה מהירה עם ממשק ה-API של OpenAI

המחברת הזו מותאמת מתוך [מאגר דוגמאות Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) הכולל מחברות שניגשות לשירותי [Azure OpenAI](aoai-assignment.ipynb).

ממשק ה-API של OpenAI בפייתון עובד גם עם דגמי Azure OpenAI, עם כמה שינויים. למידע נוסף על ההבדלים כאן: [כיצד לעבור בין נקודות קצה של OpenAI ו-Azure OpenAI עם פייתון](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# סקירה כללית  
"מודלים גדולים של שפה הם פונקציות שממפות טקסט לטקסט. בהינתן מחרוזת קלט של טקסט, מודל שפה גדול מנסה לחזות את הטקסט שיגיע אחר כך"(1). המחברת 'התחלה מהירה' הזו תציג למשתמשים מושגים ברמה גבוהה של LLM, דרישות חבילת ליבה להתחלה עם AML, הקדמה רכה לעיצוב פרומפט, וכמה דוגמאות קצרות של מקרים שונים לשימוש. 


## תוכן עניינים  

[סקירה כללית](#overview)  
[איך להשתמש בשירות OpenAI](#how-to-use-openai-service)  
[1. יצירת שירות OpenAI שלך](#1.-creating-your-openai-service)  
[2. התקנה](#2.-installation)    
[3. אישורים](#3.-credentials)  

[מקרי שימוש](#use-cases)    
[1. תמצת טקסט](#1.-summarize-text)  
[2. סווג טקסט](#2.-classify-text)  
[3. צור שמות חדשים למוצרים](#3.-generate-new-product-names)  
[4. דייק מכוון לסווג](#4.fine-tune-a-classifier)  

[הפניות](#references)


### בנה את הפקודה הראשונה שלך  
תרגיל קצר זה יספק מבוא בסיסי להגשת פקודות למודל OpenAI עבור משימה פשוטה של "סיכום".


**שלבים**:  
1. התקן את ספריית OpenAI בסביבת הפייתון שלך  
2. טען ספריות עזר סטנדרטיות והגדר את אישורי האבטחה הטיפוסיים שלך לשירות OpenAI שיצרת  
3. בחר מודל למשימה שלך  
4. צור פקודה פשוטה עבור המודל  
5. שלח את הבקשה שלך ל-API של המודל!


### 1. התקן את OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. ייבוא ספריות עזר ויצירת מופעי אישורים


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. איתור הדגם הנכון  
דגמים כמו GPT-4o ו-GPT-4o mini יכולים להבין וליצור שפה טבעית, וזמינים בפלטפורמת OpenAI עם רמות שונות של כוח ומהירות המתאימות למשימות שונות.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. עיצוב הפקודה  

"הקסם של מודלים של שפה גדולים הוא שעל ידי אימון לצמצם את טעות החיזוי הזו על כמות עצומה של טקסט, המודלים לומדים בסופו של דבר מושגים שימושיים לחיזויים אלה. לדוגמה, הם לומדים מושגים כמו"(1):

* איך לאיית
* איך דקדוק עובד
* איך לפרפרזה
* איך לענות על שאלות
* איך לנהל שיחה
* איך לכתוב בשפות רבות
* איך לתכנת
* וכו'.

#### איך לשלוט על מודל שפה גדול  
"מבין כל הקלטים למודל שפה גדול, השפעת הקלט החשובה ביותר היא טקסט הפקודה(1).

ניתן להנחות מודלים של שפה גדולים לייצר פלט בכמה דרכים:

הוראה: אמור למודל מה אתה רוצה
השלמה: גרום למודל להשלים את תחילת מה שאתה רוצה
הדגמה: הראה למודל מה אתה רוצה, עם:
כמה דוגמאות בפקודה
מאות או אלפי דוגמאות בקבוצת אימון של כוונון עדין"



#### יש שלוש הנחיות בסיסיות ליצירת פקודות:

**הראה וספר**. הבהר מה אתה רוצה באמצעות הוראות, דוגמאות, או שילוב של השניים. אם אתה רוצה שהמודל ידרג רשימת פריטים בסדר אלפביתי או יסווג פסקה לפי רגש, הראה לו שזה מה שאתה רוצה.

**ספק נתונים איכותיים**. אם אתה מנסה לבנות מסווג או לגרום למודל לעקוב אחרי דפוס, הקפד שיהיו מספיק דוגמאות. וודא לבחון את הדוגמאות שלך — המודל בדרך כלל חכם מספיק כדי לראות טעויות איות בסיסיות ולתת לך תגובה, אבל גם ייתכן ויחשוב שזו כוונה וזה עלול להשפיע על התגובה.

**בדוק את ההגדרות שלך.** הגדרות הטמפרטורה וה-top_p שולטות כמה המודל דטרמיניסטי ביצירת תגובה. אם אתה מבקש תגובה שבה יש רק תשובה נכונה אחת, תרצה להוריד את הערכים האלה. אם אתה מחפש תגובות מגוונות יותר, ייתכן ותרצה להעלות אותם. הטעות מספר אחת שאנשים עושים עם ההגדרות האלה היא להניח שאלה בקרות של "פיקחות" או "יצירתיות".


מקור: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. שלח!  


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### חזור על הקריאה הזו, איך התוצאות משתוות?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## סיכום טקסט  
#### אתגר  
סכם טקסט על ידי הוספת 'tl;dr:' בסוף קטע טקסט. שים לב כיצד המודל מבין איך לבצע מספר משימות ללא הנחיות נוספות. באפשרותך להתנסות עם בקשות תיאוריות יותר מtl;dr כדי לשנות את התנהגות המודל ולהתאים אישית את הסיכום שאתה מקבל(3).  

עבודות אחרונות הראו הישגים משמעותיים במשימות ותקנים רבים ב-NLP באמצעות אימון מקדים על מאגר טקסטים גדול ולאחר מכן כוונון עדין למשימה ספציפית. בעוד שבדרך כלל הארכיטקטורה אינה תלויה במשימה, שיטה זו עדיין דורשת מערכי נתונים מותאמים למשימה המכילים אלפי דוגמאות ואף עשרות אלפים. לעומת זאת, בני אדם בדרך כלל יכולים לבצע משימה חדשה בשפה ממספר דוגמאות מועט או מהנחיות פשוטות - דבר שמערכות NLP קיימות עדיין מתקשות לבצע במידה רבה. כאן אנו מראים שהגדלת המודל של שפה משפרת משמעותית את הביצועים במשימות בלתי תלויות, עם מעט דוגמאות, ולפעמים אפילו מגיעה לתחרותיות עם גישות כוונון עדין המובילות עד כה. 



tl;dr  


# תרגילים למספר מקרי שימוש  
1. סיכום טקסט  
2. סיווג טקסט  
3. יצירת שמות חדשים למוצרים


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## סיווג טקסט  
#### אתגר  
סווג פריטים לקטגוריות הנמסרות בזמן האינפרנס. בדוגמה הבאה, אנו מספקים גם את הקטגוריות וגם את הטקסט לסיווג בהנחייה (*playground_reference). 

פנייה של לקוח: שלום, אחת מהמפתחות במקלדת של המחשב הנייד שלי נשברה לאחרונה ואצטרך להחליף אותה:

קטגוריה מסווגת:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## צור שמות מוצרים חדשים
#### אתגר
צור שמות מוצרים ממילות דוגמה. כאן אנו כוללים בפרומפט מידע על המוצר עבורו אנו עומדים ליצור שמות. אנו גם מספקים דוגמה דומה כדי להראות את הדפוס שאנו רוצים לקבל. כמו כן, קבענו ערך טמפרטורה גבוה כדי להגדיל את האקראיות ולהשיג תגובות חדשניות יותר.

תיאור מוצר: יצרן מילקשייק ביתי
מילים בסיס: מהיר, בריא, קומפקטי.
שמות מוצרים: HomeShaker, Fit Shaker, QuickShake, Shake Maker

תיאור מוצר: זוג נעליים שיכולות להתאים לכל מידה של רגל.
מילים בסיס: מתאים, גמיש, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# מקורות  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [פורטל Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [שיטות עבודה מומלצות לכיול עדין של מודלים GPT למיון טקסט](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# לעזרה נוספת  
[צוות המסחור של OpenAI](AzureOpenAITeam@microsoft.com) 


# תורמים
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
